Getting Domain

In [ ]:
# def get_feature_domain(value:str, mode: str):
#         """
#             Look up relationships between features and domain.
        
#             Use mode = "features_to_domain" ONLY when the user provides a FEATURE NAME
#             and asks what is the domain for that feature.
        
#             Examples:
#             - "What is the domain for feature_a?"
#             - "feature a, what is the domain?"
#             - "How about feature a, what is the domain?
        
#             Use mode= "domain_to_features" when the user asks similarly like:
#             - Which features have customer domain?
#             - Which features that have customer as the domain?
           
#         """
          
    
#         value = value.lower().strip()
    
#         if mode == "features_to_domain":
    
#             value = value.lower().strip()
    
#             # -------------------------
#             # 1. Exact name search
#             # -------------------------
    
#             files = feature_index["name_exact"].get(
#                 value,
#                 []
#             )
#             if not files:
#                 files = feature_index["name_seperated"].get(
#                     value,
#                     []
#                 )
    
#             # -------------------------
#             # 2. Word search fallback
#             # -------------------------
    
#             if not files:
    
#                 words = tokenize(value)
    
#                 word_counts = Counter()
    
#                 for word in words:
    
#                     matched_files = feature_index["name_words"].get(
#                         word,
#                         []
#                     )
    
#                     for filename in matched_files:
#                         word_counts[filename] += 1
    
#                 files = [
#                     filename
#                     for filename, count
#                     in word_counts.most_common(3)
#                 ]
    
#             # -------------------------
#             # No match
#             # -------------------------
    
#             if not files:
#                 return f"Feature '{value}' not found."
    
#             # -------------------------
#             # Open best match
#             # -------------------------
    
#             filename = files[0]
    
#             yaml_path = YAML_FOLDER / filename
    
#             print("VALUE:", value)
#             print("FILES FROM INDEX:", files)
#             print("YAML_FOLDER:", YAML_FOLDER)
    
#             with open(
#                 yaml_path,
#                 "r",
#                 encoding="utf-8"
#             ) as file:
#                 data = yaml.safe_load(file)
    
#             return {
#                 "feature": data.get("name", value),
#                 "domain": data.get("domain", "")
#             }
    
    
#         elif mode == "domain_to_features":
    
#             value = value.lower().strip()
    
#             # Normalize common wording
#             value = value.replace("_", " ")
         
    
#             # Exact TTL lookup
#             files = feature_index["domain"].get(
#                 value,
#                 []
#             )
    
#             return {
#                 "domain": value,
#                 "features": files
#             }
    
#         return "Invalid mode."


Get Feature Desc

In [ ]:
def get_feature_description(filename: str):

    yaml_path = YAML_FOLDER / filename

    with open(yaml_path, "r", encoding="utf-8") as file:
        data = yaml.safe_load(file)

    return {
        "file": filename,
        "name": data.get("name", ""),
        "description": data.get("description", "")
    }

Get feature info v1

In [ ]:
def get_feature_info(filename: str):

    yaml_path = YAML_FOLDER / filename

    if not yaml_path.exists():
        return f"File not found: {filename}"

    with open(
        yaml_path,
        "r",
        encoding="utf-8"
    ) as file:

        data = yaml.safe_load(file)

    result = {
        "file": filename,
        "name": data.get("name", ""),
        "description": data.get("description", ""),
        "entity": data.get("entity", ""),
        "domain": data.get("domain", ""),
        "tags": data.get("tags", []),
        "ttl": data.get("ttl",[]),
        "feature_fields": []
    }

    for field in data.get("feature_fields", []):

        result["feature_fields"].append({
            "name": field.get("name", ""),
            "description": field.get("description", ""),
            "business_logic": field.get("business_logic", "")
        })

    return str(result)


Indexing Searching System (REAL WHOLE YAML FILE)

In [ ]:
# #Indexing 
# def search_features(query: str):
#     start_time = time.perf_counter()

#     words = query.lower().split()

#     matched_files = set()

#     # Search the feature index
#     for word in words:
#         files = feature_index.get(word, [])
#         matched_files.update(files)

#     # Load the matching YAML files
#     results = []
#     index_time  = time.perf_counter()-start_time
#     for filename in matched_files:

#         yaml_path = YAML_FOLDER / filename

#         with open(yaml_path, "r", encoding="utf-8") as file:
#             data = yaml.safe_load(file)

#         results.append({
#             "file": filename,
#             "feature": data
#         })

#     elapsed = time.perf_counter() - start_time

#     with open("search_times.txt", "a") as f:
#         f.write(
#             f"FEATURE | {query} : {elapsed:.6f} seconds\n"
#             f"Index Search | {query} : {index_time:.6f} seconds\n"
#         )

#     if not results:
#         return "No matching feature found."

#     return str(results)


# def search_features(query: str):

#     total_start = time.perf_counter()


#     #Count words per query
#     start = time.perf_counter()

#     words = query.lower().split()

#     word_counts = Counter()

#     for word in words:

#         files = feature_index.get(word, [])

#         for filename in files:
#             word_counts[filename] += 1

#     index_time = time.perf_counter() - start


    
#     #Filter top 3 only
#     ranked_files = sorted(
#         word_counts,
#         key=word_counts.get,
#         reverse=True
#     )

#     top_files = ranked_files[:2]


#     # Test
#     print("Query:", query)
#     print("Total candidates:", len(ranked_files))
#     print("Top files:")

#     for filename in top_files:
#         print(
#             filename,
#             "→",
#             word_counts[filename],
#             "matches"
#         )

    
#     results = []

   
#     for filename in top_files:

#         results.append({
#         "file": filename,
#         "matches": word_counts[filename]
#         })


#     total_time = time.perf_counter() - total_start


#     #Write Record
#     with open(
#         BASE_DIR / "search_times.txt",
#         "a",
#         encoding="utf-8"
#     ) as f:

#         f.write(
#             f"\n"
#             f"FEATURE | {query}\n"
#             f"Total Candidates | {len(ranked_files)}\n"
#             f"Files Loaded | {len(top_files)}\n"
#             f"Index Search | {index_time:.6f} seconds\n"
#             f"Total | {total_time:.6f} seconds\n"
#         )


#     if not results:
#         return "No matching feature found."

#     return str(results)


Linear Search

In [ ]:

# def search_features_2(query: str) -> str:
#     start_time = time.perf_counter()

#     yaml_folder = Path(__file__).parent / "yaml_list"
#     query = query.lower()
#     results = []

#     for yaml_path in yaml_folder.glob("*.yaml"):

#         with open(yaml_path, "r", encoding="utf-8") as file:
#             feature = yaml.safe_load(file)

#         searchable_text = flatten_yaml(feature).lower()

#         if query in searchable_text:
#             results.append({
#                 "file": yaml_path.name,
#                 "feature": feature
#             })

#     elapsed = time.perf_counter() - start_time

#     with open("search_times.txt", "a") as f:
#         f.write(f"{query} : {elapsed:.6f} seconds\n")

#     if results:
#         return str(results)

#     return "No matching feature found."


Model Prompt

In [ ]:

1. get_feature_description
   Use when the user asks what a feature is
   or asks for its description.

2. get_feature_fields
   Use when the user asks what fields,
   columns, or metrics a feature contains.

3. get_feature_entity
   Use when the user as what entity a feature contains or which features have certain entity.

4. get_feature_ttl
   Use when the users ask about ttl certain features

5. find_field_usage
    Use when the users ask about which files use the requested features

6. get_field_details
    Use this when the user asks about a particular feature field, including its description,
    business logic, or where the feature is used.

7. get_feature_domain
    Use this when the user asks questions that are related to domain to certain features.


get_ttl (solo finished)

In [ ]:
def get_feature_ttl(value: str, mode: str):
    """
        Look up relationships between features and ttl.
    
        Use mode="features_to_ttl" ONLY when the user provides a FEATURE NAME
        and asks what is the ttl for that feature.
    
        Examples:
        - "What is the ttl for feature_a?"
        - "feature a, what is the ttl?"
        - "How about feature a, what is the ttl?
    
        Use mode= "ttl_to_features" when the user asks similarly like:
        - Which features have 1000d ttl
        - 4500d ttl belongs to which features?
        - Which features have 20 years ttl?
        - Which features have 20-year ttl? 
    """
      


    if mode == "features_to_ttl":

        
        value = value.lower().strip()
        
        # Split multiple feature names
        values = re.split(
            r"\s*(?:,|\bor\b|\band\b)\s*",
                value
            )
        
        values = [
            v.strip()
            for v in values
            if v.strip()
            ]
        
        results = {}
        
        for feature in values:
        
            # Exact name
            files = feature_index["name_exact"].get(
                feature,
                []
            )
        
            # Separated name
            if not files:
                files = feature_index["name_seperated"].get(
                    feature,
                    []
                )
        
            # Word search
            if not files:
        
                words = tokenize(feature)
        
                word_counts = Counter()
        
                for word in words:
        
                    matched_files = feature_index["name_words"].get(
                        word,
                        []
                    )
        
                    for filename in matched_files:
                        word_counts[filename] += 1
        
                files = [
                    filename
                    for filename, count
                    in word_counts.most_common(3)
                ]
        
            if not files:
                results[feature] = "Feature not found."
                continue
            #Find top file
            filename = files[0]
        
            yaml_path = YAML_FOLDER / filename
        
            with open(
                yaml_path,
                "r",
                encoding="utf-8"
            ) as file:
                data = yaml.safe_load(file)
        
                results[feature] = {
                    "file": filename,
                    "entity": data.get("entity", "")
                }
        
        return results
    elif mode == "ttl_to_features":

        value = value.lower().strip()

       
        #Simple cleaning
        value = value.replace("of ttl", "")
        value = value.replace("ttl", "")
        value = value.replace("-year", " year")
        value = value.replace("-years", " years")

       
        #Find all ttl units 
        unit_match = re.search(
            r"\b(years?|days?|day)\b",
            value
        )

        if unit_match:

            unit = unit_match.group(1)
            if unit in ["year", "years"]:
                unit = "years"
            elif unit in ["days","day"]:
                unit = "d"

            # Everything before the unit
            number_part = value[:unit_match.start()]

            # Extract all numbers
            numbers = re.findall(
                r"\d+(?:\.\d+)?",
                number_part
            )

            values = [
                f"{number} {unit}"
                for number in numbers
            ]

        else:

            # No shared unit
            values = re.split(
                r"\s*(?:,|\bor\b|\band\b)\s*",
                value
            )   

            values = [
                v.strip()
                for v in values
                if v.strip()
            ]

        
        #Search index
        results = {}

        for ttl in values:

            files = feature_index["ttl"].get(
                ttl,
                []
            )

            results[ttl] = files

        return {
            "requested_ttls": values,
            "features": results
        }
    return "Invalid Mode"


get feature_field

In [ ]:
def get_feature_fields(value: str, mode: str):
    """
        Look up relationships between features and feature_fields.
    
        Use mode="features_to_feature_fields" ONLY when the user provides a FEATURE NAME
        and asks what are the feature fields the features contain.
    
        Examples:
        - What feature fields featureA contains?
        - featureA and featureB have what feature_fields?
        - What feature fields does featureA contain?
    
 
    """
      

    value = value.lower().strip()

    if mode == "features_to_feature_fields":            
                
        value = value.lower().strip()
                
        # Split multiple feature names
        values = re.split(
            r"\s*(?:,|\bor\b|\band\b)\s*",
            value
            )
                
        values = [
            v.strip()
            for v in values
            if v.strip()
            ]
                
        results = {}
                
        for feature in values:
                
            # Exact name
            files = feature_index["name_exact"].get(
                feature,
                []
            )
                
            # Separated name
            if not files:
                files = feature_index["name_seperated"].get(
                    feature,
                    []
                )
                
            # Word search
            if not files:
                
                words = tokenize(feature)
                
                word_counts = Counter()
                
                for word in words:
                
                    matched_files = feature_index["name_words"].get(
                        word,
                        []
                    )
                
                    for filename in matched_files:
                        word_counts[filename] += 1
                
                    files = [
                        filename
                        for filename, count
                        in word_counts.most_common(3)
                    ]
                
            if not files:
                results[feature] = "Feature not found."
                continue
            #Find top file
            filename = files[0]
                
            yaml_path = YAML_FOLDER / filename
                
            with open(
                yaml_path,
                "r",
                encoding="utf-8"
            ) as file:
                data = yaml.safe_load(file)
                
                results[feature] = {
                    "file": filename,
                        "feature_fields": [
                            field.get("name", "")
                            for field in data.get("feature_fields", [])
                        ]
                    }
                
            return results

    elif mode == "feature_fields_to_features":

        value = value.lower().strip()

        # Normalize common wording
        value = value.replace("_", " ")
        

        # Exact TTL lookup
        files = feature_index["ttl"].get(
            value,
            []
        )

        return {
            "ttl": value,
            "features": files
        }

    return "Invalid mode."


get entity (solo finished)

In [ ]:
def get_feature_entity(value: str, mode: str):
    """
    Look up relationships between features and entities.

    Use mode="feature_to_entity" ONLY when the user provides a FEATURE NAME
    and asks which ENTITY that feature uses.

    Examples:
    - "What entity does feature A use?"
    - "What is the entity for feature_a?"
    - "Which entity belongs to contract_basic_info?"

    Do NOT use this tool when the user asks which
    features use an entity.

    Use mode="entity_to_features" when the user asks similarly like:
    - Which features use the customer entity?
    - What features belong to the customer entity?
    """
    value = value.lower().strip()

    if mode == "feature_to_entity":

        value = value.lower().strip()
        
        # Split multiple feature names
        values = re.split(
            r"\s*(?:,|\bor\b|\band\b)\s*",
                value
            )
        
        values = [
            v.strip()
            for v in values
            if v.strip()
            ]
        
        results = {}
        
        for feature in values:
        
            # Exact name
            files = feature_index["name_exact"].get(
                feature,
                []
            )
        
            # Separated name
            if not files:
                files = feature_index["name_seperated"].get(
                    feature,
                    []
                )
        
            # Word search
            if not files:
        
                words = tokenize(feature)
        
                word_counts = Counter()
        
                for word in words:
        
                    matched_files = feature_index["name_words"].get(
                        word,
                        []
                    )
        
                    for filename in matched_files:
                        word_counts[filename] += 1
        
                files = [
                    filename
                    for filename, count
                    in word_counts.most_common(3)
                ]
        
            if not files:
                results[feature] = "Feature not found."
                continue
            #Find top file
            filename = files[0]
        
            yaml_path = YAML_FOLDER / filename
        
            with open(
                yaml_path,
                "r",
                encoding="utf-8"
            ) as file:
                data = yaml.safe_load(file)
        
                results[feature] = {
                    "file": filename,
                    "entity": data.get("entity", "")
                }
        
        return results
        

    elif mode == "entity_to_features":

        value = value.lower().strip()

        values = re.split(
            r"\s*(?:,|\bor\b|\band\b)\s*",
            value
        )

        values = [
            v.strip()
            for v in values
            if v.strip()
        ]

        results = {}

        for entity in values:

            files = feature_index["entity"].get(
                entity,
                []
            )

            results[entity] = files

        return {
            "requested_entities": values,
            "features": results
        }
    
    return "Invalid mode."


Get field details (linear search)

In [ ]:
def get_field_details(filename: str, field_name: str):
    """
    Get ONLY information about one specific feature field.

    Use this when the user asks about a particular
    feature field, including its description,
    business logic, or where the feature is used.
    """

    yaml_path = YAML_FOLDER / filename

    if not yaml_path.exists():
        return f"File not found: {filename}"

    with open(yaml_path, "r", encoding="utf-8") as file:
        data = yaml.safe_load(file)

    # Find the requested field
    for field in data.get("feature_fields", []):

        if field.get("name", "").lower() == field_name.lower():

            return {
                "feature": data.get("name", ""),
                "field": {
                    "name": field.get("name", ""),
                    "description": field.get("description", ""),
                    "business_logic": field.get("business_logic", "")
                },
                "feature_usage": {
                    "entity": data.get("entity", ""),
                    "domain": data.get("domain", ""),
                    "tags": data.get("tags", [])
                }
            }

    return f"Field '{field_name}' not found in {filename}."


Search SQL Linear

In [ ]:
# def search_sql(query: str):
#     start_time = time.perf_counter()

#     words = query.lower().split()

#     matched_files = set()

#     # Search the SQL index
#     for word in words:
#         files = sql_index.get(word, [])
#         matched_files.update(files)

#     # Load the matching YAML files
#     results = []

#     for filename in matched_files:

#         yaml_path = YAML_FOLDER / filename

#         with open(yaml_path, "r", encoding="utf-8") as file:
#             data = yaml.safe_load(file)

#         results.append({
#             "file": filename,
#             "feature": data
#         })

#     elapsed = time.perf_counter() - start_time

#     with open("search_times.txt", "a") as f:
#         f.write(
#             f"SQL | {query} : {elapsed:.6f} seconds\n"
#         )

#     if not results:
#         return "No matching feature found."

#     return str(results)


Search SQL (linear)

In [ ]:
def search_sql(query: str):

    total_start = time.perf_counter()

    #Count words per query
    start = time.perf_counter()

    words = query.lower().split()

    word_counts = Counter()

    for word in words:

        files = sql_index.get(word, [])

        for filename in files:
            word_counts[filename] += 1

    index_time = time.perf_counter() - start

    #Filter top 3 only
    ranked_files = sorted(
        word_counts,
        key=word_counts.get,
        reverse=True
    )


    top_files = ranked_files[:3]


    # Test
    print("SQL Query:", query)
    print("Total candidates:", len(ranked_files))
    print("Top files:")


    for filename in top_files:
        print(
            filename,
            "=",
            word_counts[filename],
            "matches"
        )

    #Read top 3 files only
    results = []
    read_time_total = 0
    parse_time_total = 0

    for filename in top_files:
    
            results.append({
            "file": filename,
            "matches": word_counts[filename]
            })
    
    
    total_time = time.perf_counter() - total_start


    
#Write record
    with open(
        BASE_DIR / "search_times.txt",
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            f"\n"
            f"SQL | {query}\n"
            f"Total Candidates | {len(ranked_files)}\n"
            f"Files Loaded | {len(top_files)}\n"
            f"Index Search | {index_time:.6f} seconds\n"
            f"File Reading | {read_time_total:.6f} seconds\n"
            f"YAML Parsing | {parse_time_total:.6f} seconds\n"
            f"Total | {total_time:.6f} seconds\n"
        )


    if not results:
        return "No matching feature found."

    return str(results)



Find features field

In [ ]:
def find_field_usage(field_name: str):

    field_name = field_name.lower()

    files = feature_index.get(field_name, [])

    if not files:
        return f"No feature uses field '{field_name}'."

    return {
        "field": field_name,
        "used_by": files
    }


Flattening YAML

In [ ]:
def flatten_yaml(value):
    if isinstance(value, dict):
        return " ".join(
            flatten_yaml(v)
            for v in value.values()
        )

    if isinstance(value, list):
        return " ".join(
            flatten_yaml(item)
            for item in value
        )

    return str(value)


Migrate entities embeddings to chromaDB

In [ ]:
import json
from pathlib import Path

import chromadb
import yaml

# CONFIG

BASE_DIR = Path(__file__).resolve().parents[1]

EMBEDDING_FILE = BASE_DIR / "entities_description_embeddings.json"
ENTITY_FOLDER = BASE_DIR / "entities"
CHROMA_FOLDER = BASE_DIR / "chromadb"

COLLECTION_NAME = "entities_description_bl_collection"


# LOAD EXISTING EMBEDDINGS
with open(EMBEDDING_FILE, "r", encoding="utf-8") as file:
    entity_embeddings = json.load(file)

# CONNECT TO CHROMADB
chroma_client = chromadb.PersistentClient(path=str(CHROMA_FOLDER))


# GET OR CREATE COLLECTION
try:
    collection = chroma_client.get_collection(name=COLLECTION_NAME)
    print(f"Using existing collection: {COLLECTION_NAME}")

except Exception:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        configuration={
            "hnsw": {
                "space": "cosine"
            }
        }
    )

    print(f"Created collection: {COLLECTION_NAME}")


# MIGRATE EMBEDDINGS
migrated = 0
skipped = 0

for yaml_filename, embedding in entity_embeddings.items():
    yaml_path = ENTITY_FOLDER / yaml_filename
    # Check YAML exists
    if not yaml_path.exists():
        print(f"WARNING: Entity YAML not found: {yaml_path}")
        skipped += 1
        continue


    # Check if this embedding already exists in Chroma
    existing = collection.get(
        ids=[yaml_filename],
        include=[]
    )

    if existing["ids"]:
        print("Already exists in Chroma. Skipping.")
        skipped += 1
        continue


    # Load entity YAML
    with open(yaml_path,"r",encoding="utf-8") as file:
        entity_data = yaml.safe_load(file)


    if not entity_data:
        skipped += 1
        continue


    # Extract entity information
    entity_name = entity_data.get("name")
    description = entity_data.get("description") 
    business_logic = entity_data.get("business_logic") 


    embedding_text = f"""
        Description: {description}
        Business Logic: {business_logic}
        """.strip()


    # Metadata
    metadata = {
        "entity": entity_name,
        "yaml_file": yaml_filename,
        "description": description,
        "business_logic": business_logic,
    }


    # Save to Chroma
    collection.upsert(
        ids=[yaml_filename],
        embeddings=[embedding],
        documents=[embedding_text],
        metadatas=[metadata],
    )

    migrated += 1

# ============================================================
# FINAL SUMMARY
# ============================================================



print(f"Embeddings in JSON : {len(entity_embeddings)}")
print(f"Chroma count        : {collection.count()}")
print(f"Collection          : {COLLECTION_NAME}")
print(f"Chroma location     : {CHROMA_FOLDER.resolve()}")

Normalize Terminology

In [ ]:
import json
from pathlib import Path

INPUT_FILE = Path("terminology_index.json")
OUTPUT_FILE = Path("terminology_index_normalized.json")


with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


for item in data:

    if "term" in item and item["term"]:
        item["term"] = item["term"].lower()

    if "definition" in item and item["definition"]:
        item["definition"] = item["definition"].lower()


with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        indent=2,
        ensure_ascii=False
    )


print(f"Saved normalized terminology index to: {OUTPUT_FILE}")